# OrCaScore — concordância arterial entre IA e pipeline

Compara a máscara final das artérias coronárias do pipeline com um modelo de
IA em resolução `mid`. O Dice mede **concordância entre as previsões** no mesmo
exame CTAI. Uma concordância alta também pode ocorrer quando os métodos
cometem o mesmo erro anatômico.

O pipeline permanece na grade em que rodou. A máscara da IA é alinhada pelo
affine NIfTI com vizinho mais próximo, quando necessário.

In [ ]:
# ruff: noqa: E402
import getpass
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break
else:
    raise RuntimeError("Abra o Jupyter na raiz do projeto ou em uma subpasta dele.")

from utils.comparison_utils.orcascore_artery_agreement import (
    build_orcascore_pairs,
    compare_orcascore_pairs,
    summarize_orcascore_agreement,
    summarize_orcascore_dice_histogram,
)
from utils.project.runtime.notebook_env import (
    configure_notebook_environment,
    resolve_existing_path,
)

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)

In [ ]:
from eda.comparisons.view.orcascore_artery_agreement_views import plot_dice_histogram

## Seleção do modelo e do run

Defina `AI_CHANNELS = 12` ou `AI_CHANNELS = 4`. Cada execução lê apenas o
modelo `mid` selecionado. O alinhamento usa a geometria gravada nos arquivos.
Deixe `EXAM_IDS` vazio para incluir toda a seleção do run, ou informe IDs para
uma conferência rápida. Use `CCTA_RESULTS_ROOT` se a pasta de resultados
estiver em outro disco.

In [ ]:
PIPELINE_RESOLUTION = "mid"
PIPELINE_RUN = "2026-11-20_19-07-17"
AI_EXPORT = "LabelArteriasOrcaScore"
AI_CHANNELS = 12  # Opções: 4 ou 12.
EXAM_IDS = ()

if AI_CHANNELS not in (4, 12):
    raise ValueError("Escolha 4 ou 12 canais para o modelo mid.")
AI_VARIANT = f"mid_resolution_{AI_CHANNELS}_channels"

MOUNT_ROOT = Path("/run/media") / getpass.getuser()
RESULTS_ROOT = resolve_existing_path(
    "CCTA_RESULTS_ROOT",
    sorted(MOUNT_ROOT.glob("*/Results_dataset_ccta")),
    "raiz de resultados CCTA",
)
ORCASCORE_ROOT = RESULTS_ROOT / "orcascore"
RUN_DIR = ORCASCORE_ROOT / f"{PIPELINE_RESOLUTION}_res" / PIPELINE_RUN
AI_ROOT = ORCASCORE_ROOT / "ai_predictions" / "artery" / AI_EXPORT

## Pareamento e Dice

O manifesto do run define os exames e o subconjunto. Cada ID é pareado com
`<exam_id>CTAI/pre_label.nii.gz` do modelo escolhido. Máscaras ausentes ou
inválidas ficam sem Dice. Se apenas uma das máscaras na grade comum estiver
vazia, o Dice é zero. Se ambas estiverem vazias, o Dice fica indisponível.

In [ ]:
pairs = build_orcascore_pairs(RUN_DIR, AI_ROOT, (AI_VARIANT,))
if EXAM_IDS:
    missing_ids = set(EXAM_IDS).difference(pairs["exam_id"])
    if missing_ids:
        raise ValueError(f"Exames ausentes da seleção do run: {sorted(missing_ids)}")
    pairs = pairs.loc[pairs["exam_id"].isin(EXAM_IDS)]

results = compare_orcascore_pairs(pairs)
summary = summarize_orcascore_agreement(results, results)

## Dice em treino, teste e no geral

As estatísticas usam os exames com Dice definido. A linha **Geral** reúne
treino e teste com o mesmo peso por exame. As contagens de cobertura mostram
quantos exames entraram em cada média.

In [ ]:
columns = {
    "subset": "Conjunto",
    "selected": "Exames selecionados",
    "pipeline_masks": "Máscaras do pipeline",
    "ai_masks": "Máscaras da IA",
    "dice_available": "Exames com Dice",
    "dice_mean": "Dice médio",
    "dice_median": "Dice mediano",
    "dice_std": "Desvio padrão",
}
display(
    summary.loc[:, list(columns)]
    .replace({"subset": {"train": "Treino", "test": "Teste"}})
    .rename(columns=columns)
    .round(3)
)

## Dice por exame

A tabela mostra cada par e o estado da comparação. Em caso de ausência ou
falha, consulte os campos de estado e erro. Os volumes são medidos na mesma
grade, após o alinhamento da IA.

In [ ]:
fields = [
    "subset",
    "exam_id",
    "dice",
    "status",
    "pipeline_volume_ml",
    "ai_volume_ml",
    "pipeline_error",
    "detail",
]
with pd.option_context("display.max_rows", None):
    display(results.loc[:, fields].sort_values(["subset", "exam_id"]).round(3))

## Histograma de Dice

A tabela e o gráfico mostram as mesmas frequências em intervalos de 0,1.
O limite inferior é inclusivo. O último intervalo inclui o Dice igual a 1.

In [ ]:
histogram = summarize_orcascore_dice_histogram(results)
display(histogram.round(1))
plot_dice_histogram(histogram)
plt.show()

## Sobreposição das artérias em K3D

As três cenas usam a tabela de Dice do modelo selecionado. Cada cena sobrepõe
as artérias e mostra os óstios registrados pelo pipeline. Para acrescentar
aorta, associe o ID do exame à sua máscara binária na mesma grade em
`AORTA_MASK_PATHS`.

In [ ]:
from eda.comparisons.view.orcascore_artery_agreement_views import (
    show_orcascore_dice_case,
)
from utils.comparison_utils.orcascore_artery_agreement import select_orcascore_dice_case

AORTA_MASK_PATHS = {}  # Opcional: mapeie o ID do exame ao caminho do NIfTI da aorta.

### Maior Dice geral

Usa o maior valor entre todos os exames do modelo selecionado.

In [ ]:
best_case = select_orcascore_dice_case(results)
show_orcascore_dice_case(best_case, AORTA_MASK_PATHS)

### Dice perto de 0,60

Prioriza `TEV1P1` se ele estiver a até 0,01 de 0,60. Caso contrário,
seleciona o exame com Dice mais próximo desse valor.

In [ ]:
case_60 = select_orcascore_dice_case(results, 0.60, preferred_exam_id="TEV1P1")
show_orcascore_dice_case(case_60, AORTA_MASK_PATHS)

### Dice perto de 0,50

Seleciona o exame com Dice mais próximo de 0,50.

In [ ]:
case_50 = select_orcascore_dice_case(results, 0.50)
show_orcascore_dice_case(case_50, AORTA_MASK_PATHS)

In [ ]:
case_zero_1 = select_orcascore_dice_case(
    results.loc[results["dice"].eq(0) & results["exam_id"].eq("TRV1P7")]
)
show_orcascore_dice_case(case_zero_1, AORTA_MASK_PATHS)

In [ ]:
case_zero_2 = select_orcascore_dice_case(
    results.loc[results["dice"].eq(0) & results["exam_id"].eq("TRV3P7")]
)
show_orcascore_dice_case(case_zero_2, AORTA_MASK_PATHS)

In [ ]:
case_zero_3 = select_orcascore_dice_case(
    results.loc[results["dice"].eq(0) & results["exam_id"].eq("TEV3P1")]
)
show_orcascore_dice_case(case_zero_3, AORTA_MASK_PATHS)